# Using MLP approach to create character level model

Inspiration: MLP based approach for a Word Level Model

1. Learning representation of token representation (here, characters; in research paper it was words)
2. Simulatenously, learning the model parameters

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 

%matplotlib inline

In [ ]:
data = open("../data.txt", 'r').read().splitlines()

In [ ]:
stoi = {chr(97+i):i+1 for i in range(26)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [ ]:
# building dataset

context_size = 3    # context/block length
X, y = [], []

for w in data:

    context = [0] * context_size
    #print(w)
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])

        context = context[1:] + [ix]    # crop and append

X = torch.tensor(X)
y = torch.tensor(y)

In [ ]:
X.shape, X.dtype, y.shape, y.dtype

In [ ]:
# Lookup table for character representation by condensing them into lower dimension to prevent the dimension explosion problem
dimensions = 2

C = torch.randn((27, dimensions))    # 27 represents the vocabulary here 27 characters; 2 represents the number of dimensions we want to use to represent the characters
# in the RP: 17000 words were compressed into m dimensions, ie, (17000, m) lookup table

In [ ]:
# embedding for the dataset

emb = C[X]
emb.shape

#emb.shape = (x, y, z) ---> x: no of rows of training set; y = size of context; z = dimensions

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility

In [ ]:
# neural network
# number of input to the network = context_size * dimensions for each neuron

layer1size = 100

W1 = torch.randn(((context_size * dimensions), layer1size), generator=g)
b1 = torch.randn(layer1size, generator=g)

# to perform: emb @ W1 + b1
# therefore we need to change the shape of emb to (x, y*z)

# torch.concat(torch.unbind(emb, 1), 1)

emb = emb.view(emb.shape[0], context_size*dimensions)

# output layer parameters

W2 = torch.randn((layer1size, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [ ]:
h = torch.tanh(emb @ W1 + b1)   # first hidden layer
logits = h @ W2 + b2    # output layer

""" counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss = -probs[torch.arange(emb.shape[0]), y].log().mean()   # select the probability of the correct prediction(y) given by the current model
 """
loss = F.cross_entropy(logits, y)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lr = 0.1

for _ in range(10000):

    # mini batch
    randomindex = torch.randint(0, X.shape[0], (32,))   #32 here is batch size
    X_batch = X[randomindex]
    y_batch = y[randomindex]
    # forward pass
    emb = C[X_batch]
    h = torch.tanh(emb.view(-1, context_size*dimensions) @ W1 + b1)   # first hidden layer
    logits = h @ W2 + b2    # output layer
    loss = F.cross_entropy(logits, y_batch)

    if(_ % 1000 == 0):    print(loss.item())

    #backward pass
    for p in parameters:
        p.grad = None

    loss.backward()

    # update
    for p in parameters:
        p.data += -lr * p.grad


## Using train, val, test split

In [ ]:
def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * context_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append


  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(data)

n1 = int(0.8*len(data))
n2 = int(0.9*len(data))

X_train, y_train = build_dataset(data[:n1])
X_val, y_val = build_dataset(data[n1:n2])
X_test, y_test = build_dataset(data[n2:])

In [ ]:
dimensions = 10
C = torch.randn((27, dimensions), generator=g)
emb = C[X]
emb.shape

In [ ]:
layer1size = 200

W1 = torch.randn(((context_size * dimensions), layer1size), generator=g)
b1 = torch.randn(layer1size, generator=g)

W2 = torch.randn((layer1size, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # number of parameters in total

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lr = 0.1

for _ in range(200000):

    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (32,))
  
    # forward pass
    emb = C[X_train[ix]] # (32, 3, 10)
    h = torch.tanh(emb.view(-1, context_size*dimensions) @ W1 + b1)   # first hidden layer
    logits = h @ W2 + b2    # output layer
    loss = F.cross_entropy(logits, y_train[ix])

    if(_ % 10000 == 0):    print(loss.item())

    #backward pass
    for p in parameters:
        p.grad = None

    loss.backward()

    # update
    lr = 0.1 if lr<100000 else 0.01
    for p in parameters:    #missed the loop that was why the error was not converging as not all the parameters were being updated
        p.data += -lr * p.grad


print(loss.item())

In [ ]:
# using val datatset to test

emb = C[X_val]
h = torch.tanh(emb.view(-1, (context_size*dimensions)) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, y_val)

loss

## Visualizing the embedding

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters

plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * context_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

### Note: The code gives a massive loss resulting in poor sampling likely due to the parameter and hyperparameter settings for this particular code, though architectural nature of the model is the same as Andrej's tutorial

for p in parameters: p.data += -lr * p.grad

#missed the loop that was why the error was not converging as not all the parameters were being updated

Silly mistake tbh ;)